#  Pharma Sales Performance Analysis
> *Regional sales tracking, product performance, and target achievement across a pharmaceutical portfolio.*

**Dataset:** 200 sales records | 5 products | 5 regions | 10 sales reps  
**Tools:** Python · Pandas · Matplotlib · Seaborn · SQL (queries in `/queries/`)  
**Business Question:** *Which products, regions and reps are driving — or missing — targets?*

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('data/pharma_sales_sample.csv')
print(f" Loaded {len(df)} records")
print(f"Products: {df['product'].unique()}")
print(f"Regions:  {df['region'].unique()}")

In [ ]:
# KPI Summary
print("=" * 45)
print("         SALES PERFORMANCE DASHBOARD")
print("=" * 45)
print(f"  Total Revenue Generated:  €{df['actual_sales'].sum()/1e6:.2f}M")
print(f"  Total Target:             €{df['sales_target'].sum()/1e6:.2f}M")
print(f"  Overall Achievement:       {df['actual_sales'].sum()/df['sales_target'].sum()*100:.1f}%")
print(f"  Avg Achievement per Rep:   {df['achievement_pct'].mean():.1f}%")
print(f"  Total New Customers:       {df['new_customers'].sum():,}")
print("=" * 45)

##  Product Performance

In [ ]:
# Sales by product
prod = df.groupby('product').agg(
    total_sales=('actual_sales','sum'),
    avg_achievement=('achievement_pct','mean'),
    total_units=('units_sold','sum')
).sort_values('total_sales', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13,4))

# Chart 1: Total sales
axes[0].bar(prod.index, prod['total_sales']/1000,
            color=sns.color_palette("Blues_d", len(prod)))
axes[0].set_title('Total Sales by Product (€K)', fontweight='bold')
axes[0].set_ylabel('Sales (€K)')

# Chart 2: Achievement %
colors_ach = ['#2ecc71' if v>=100 else '#f39c12' if v>=80 else '#e74c3c'
              for v in prod['avg_achievement']]
axes[1].bar(prod.index, prod['avg_achievement'], color=colors_ach)
axes[1].axhline(100, color='navy', linestyle='--', alpha=0.6, label='Target 100%')
axes[1].set_title('Avg Target Achievement by Product (%)', fontweight='bold')
axes[1].set_ylabel('Achievement (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('product_performance.png', dpi=150, bbox_inches='tight')
plt.show()

##  Regional Heatmap

In [ ]:
# Region × Product heatmap
pivot = df.pivot_table(values='achievement_pct', index='region',
                        columns='product', aggfunc='mean')

fig, ax = plt.subplots(figsize=(11,5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn',
            center=100, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Achievement %'})
ax.set_title('Target Achievement (%) — Region × Product', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('region_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n💡 Green = over target | Red = under target")

## 🏆 Top Sales Reps

In [ ]:
# Rep leaderboard
reps = df.groupby('sales_rep').agg(
    total_sales=('actual_sales','sum'),
    avg_achievement=('achievement_pct','mean'),
    new_customers=('new_customers','sum')
).sort_values('avg_achievement', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10,5))
colors = ['gold' if i==0 else 'silver' if i==1 else '#cd7f32' if i==2 else '#5b9bd5'
          for i in range(len(reps))]
ax.barh(reps.index[::-1], reps['avg_achievement'][::-1], color=colors[::-1])
ax.axvline(100, color='red', linestyle='--', alpha=0.5, label='Target')
ax.set_title('Sales Rep Leaderboard — Avg Achievement %', fontweight='bold')
ax.set_xlabel('Achievement (%)')
ax.legend()
plt.tight_layout()
plt.savefig('rep_leaderboard.png', dpi=150, bbox_inches='tight')
plt.show()

## 💼 Key Business Insights

| Finding | Recommendation |
|---|---|
| Some products consistently miss targets | Review pricing strategy or sales training for underperformers |
| Regional variation is significant | Investigate whether it's market size or rep performance |
| Top reps outperform by 30–40% | Document and share best practices company-wide |

📂 **SQL queries** for deeper analysis in `/queries/sales_analysis.sql`

---
*Analysis by [Your Name] · Data Analyst Portfolio Project*